# Pneumonia CNN — ONNX Deployment & Model Monitoring Pipeline
 
 
## Overview
 
This notebook deploys a pneumonia detection CNN to a SageMaker real time endpoint using ONNX Runtime, then implements a full monitoring stack covering model quality, data quality, and infrastructure metrics via CloudWatch.
 
**Problem Solved:** The original model was trained with TensorFlow 2.19 / Keras 3.14 (Python 3.12). No SageMaker inference container available in the AWS Academy lab account could load a Keras 3–saved model cleanly — every container either used Keras 2 (incompatible format) or a different Keras 3 minor version (incompatible BatchNorm args). Converting to ONNX eliminated all framework version dependencies.
 

##  ONNX Conversion & Endpoint Deployment

In [7]:
!pip install tf2onnx onnxruntime --quiet

In [8]:
# Convert Keras model to ONNX 
 
import numpy as np
import tensorflow as tf
import tf2onnx
 
# Load your existing model (works fine locally — the problem was only on the *inference container*)
model = tf.keras.models.load_model("best_model.keras")
print(f"Model loaded. Input shape: {model.input_shape}, Output shape: {model.output_shape}")
 
# Define input signature — your model expects (batch, 128, 128, 1)
input_signature = [tf.TensorSpec(shape=(None, 128, 128, 1), dtype=tf.float32, name="input")]
 
# Convert to ONNX
model_proto, _ = tf2onnx.convert.from_keras(
    model,
    input_signature=input_signature,
    opset=13,  # widely supported opset version
    output_path="model.onnx"
)
print("Saved model.onnx")

Model loaded. Input shape: (None, 128, 128, 1), Output shape: (None, 1)


I0000 00:00:1781494416.719483     578 devices.cc:76] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0 (Note: TensorFlow was not compiled with CUDA or ROCm support)
I0000 00:00:1781494416.719790     578 single_machine.cc:374] Starting new session


I0000 00:00:1781494417.080886     578 devices.cc:76] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0 (Note: TensorFlow was not compiled with CUDA or ROCm support)
I0000 00:00:1781494417.081123     578 single_machine.cc:374] Starting new session


Saved model.onnx


In [37]:
# Verify ONNX model locally 
 
import onnxruntime as ort
from PIL import Image
import os, glob
 
session = ort.InferenceSession("model.onnx")
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name
 
print(f"ONNX input:  {input_name}, shape {session.get_inputs()[0].shape}")
print(f"ONNX output: {output_name}, shape {session.get_outputs()[0].shape}")
 
# Quick sanity check with a real image
test_images = glob.glob("/home/sagemaker-user/preprocessed-images/*.png")[:5]
for img_path in test_images:
    img = Image.open(img_path).convert("L").resize((128, 128))
    arr = np.array(img, dtype=np.float32) 
    arr = arr.reshape(1, 128, 128, 1)
    pred = session.run([output_name], {input_name: arr})[0]
    label = "PNEUMONIA" if pred[0][0] >= 0.4 else "NORMAL"
    print(f"  {os.path.basename(img_path)}: prob={pred[0][0]:.4f} → {label}")
 
print("\nONNX model verified — predictions match expected format.")

ONNX input:  input, shape ['unk__145', 128, 128, 1]
ONNX output: output, shape ['unk__146', 1]
  preprocessed-images_NORMAL_IM-0045-0001.png: prob=0.0454 → NORMAL
  preprocessed-images_NORMAL_IM-0001-0001.png: prob=0.0698 → NORMAL
  preprocessed-images_NORMAL_IM-0046-0001.png: prob=0.1059 → NORMAL
  preprocessed-images_NORMAL_IM-0003-0001.png: prob=0.1191 → NORMAL
  preprocessed-images_NORMAL_IM-0049-0001.png: prob=0.4027 → PNEUMONIA

ONNX model verified — predictions match expected format.


In [38]:
# Write inference.py for SageMaker 

inference_code = '''
import onnxruntime as ort
import numpy as np
from PIL import Image
import io, json, os
 
def model_fn(model_dir):
    """Load the ONNX model."""
    model_path = os.path.join(model_dir, "model.onnx")
    session = ort.InferenceSession(model_path)
    return session
 
def input_fn(request_body, request_content_type):
    """Preprocess: accept raw image bytes."""
    if request_content_type in ("image/png", "image/jpeg", "application/x-image"):
        img = Image.open(io.BytesIO(request_body)).convert("L").resize((128, 128))
        arr = np.array(img, dtype=np.float32) 
        arr = arr.reshape(1, 128, 128, 1)
        return arr
    elif request_content_type == "application/json":
        data = json.loads(request_body)
        arr = np.array(data["instances"], dtype=np.float32)
        if arr.ndim == 3:
            arr = arr.reshape(1, 128, 128, 1)
        return arr
    else:
        raise ValueError(f"Unsupported content type: {request_content_type}")
 
def predict_fn(input_data, session):
    """Run ONNX inference."""
    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name
    result = session.run([output_name], {input_name: input_data})
    return result[0]
 
def output_fn(prediction, accept):
    """Format output as JSON."""
    prob = float(prediction[0][0])
    label = "PNEUMONIA" if prob >= 0.4 else "NORMAL"
    response = {
        "probability": prob,
        "predicted_label": label,
        "threshold": 0.4
    }
    return json.dumps(response), "application/json"
'''
 
os.makedirs("onnx_code", exist_ok=True)
with open("onnx_code/inference.py", "w") as f:
    f.write(inference_code)
print("Written: onnx_code/inference.py")

Written: onnx_code/inference.py


In [39]:
# Write requirements.txt 

with open("onnx_code/requirements.txt", "w") as f:
    f.write("onnxruntime\nPillow\nnumpy\n")
print("Written: onnx_code/requirements.txt")

Written: onnx_code/requirements.txt


In [40]:
# Package model.tar.gz and upload to S3 
 
import tarfile, boto3, sagemaker
 
# Package: model.onnx at root, inference.py + requirements.txt 
with tarfile.open("model_onnx.tar.gz", "w:gz") as tar:
    tar.add("model.onnx", arcname="model.onnx")
    tar.add("onnx_code/inference.py", arcname="code/inference.py")
    tar.add("onnx_code/requirements.txt", arcname="code/requirements.txt")
print("Created model_onnx.tar.gz")
 
# Verify contents
with tarfile.open("model_onnx.tar.gz", "r:gz") as tar:
    tar.list()
 
# Upload to S3
bucket = "pneumonia-data-set-group-4"
s3_model_path = f"s3://{bucket}/models/model_onnx.tar.gz"
boto3.client("s3").upload_file("model_onnx.tar.gz", bucket, "models/model_onnx.tar.gz")
print(f"Uploaded to {s3_model_path}")

Created model_onnx.tar.gz
?rw-r--r-- sagemaker-user/users    5036069 2026-06-15 03:33:37 model.onnx 
?rw-r--r-- sagemaker-user/users       1542 2026-06-15 04:16:47 code/inference.py 
?rw-r--r-- sagemaker-user/users         25 2026-06-15 04:16:49 code/requirements.txt 


Uploaded to s3://pneumonia-data-set-group-4/models/model_onnx.tar.gz


In [16]:
# Deploy to SageMaker endpoint 

from sagemaker.sklearn import SKLearnModel
from sagemaker import get_execution_role
from datetime import datetime
 
role = get_execution_role()   # LabRole
timestamp = datetime.now().strftime("%Y-%m-%d-%H%M")
 
onnx_model = SKLearnModel(
    model_data=s3_model_path,
    role=role,
    framework_version="1.2-1",
    py_version="py3",
    entry_point="onnx_code/inference.py",       
    dependencies=["onnx_code/requirements.txt"],
)
 
endpoint_name = f"pneumonia-onnx-{timestamp}"
print(f"Deploying endpoint: {endpoint_name}")
print("This takes 5-8 minutes...")
 
predictor = onnx_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=endpoint_name,
)

print(f"Endpoint {endpoint_name} is InService!")

Deploying endpoint: pneumonia-onnx-2026-06-15-0338
This takes 5-8 minutes...


-

-

-

-

-

-

!

Endpoint pneumonia-onnx-2026-06-15-0338 is InService!


In [17]:
# Test the live endpoint 
 
import json
 
# Read a test image
test_img_path = test_images[0]
with open(test_img_path, "rb") as f:
    img_bytes = f.read()
 
# Invoke endpoint with raw image bytes
predictor.serializer = sagemaker.serializers.IdentitySerializer(content_type="image/png")
predictor.deserializer = sagemaker.deserializers.JSONDeserializer()
 
response = predictor.predict(img_bytes)
print(f"Endpoint response: {json.dumps(response, indent=2)}")
print(f"\nImage: {os.path.basename(test_img_path)}")
print(f"Probability: {response['probability']:.4f}")
print(f"Prediction:  {response['predicted_label']}")

Endpoint response: {
  "probability": 0.7699844241142273,
  "predicted_label": "PNEUMONIA",
  "threshold": 0.4
}

Image: preprocessed-images_NORMAL_IM-0045-0001.png
Probability: 0.7700
Prediction:  PNEUMONIA


# Monitoring Pipeline Overview
 
## What We Monitor
 
Our pipeline monitors the deployed pneumonia CNN endpoint across three layers:

**Model Quality** — tracks whether the model's predictions are still accurate. We compute accuracy (0.816), precision (0.699), recall (0.747), F1 (0.722), and AUC (0.877) by comparing predictions against ground truth labels. SageMaker's Model Quality Monitor generates baseline statistics and constraint thresholds. If future metrics fall below these thresholds, we know the model is degrading.
 
**Data Quality** — tracks whether incoming images still look like the training data. We measure pixel level statistics (mean, std, min, max, median) across the image set. SageMaker's Data Quality Monitor establishes the expected distributions. If new images arrive with significantly different brightness, contrast, or resolution, the monitor flags data drift.
 
**Infrastructure** — tracks whether the endpoint is healthy. AWS automatically logs invocations, errors, latency, CPU, memory, and disk usage. Six CloudWatch alarms fire if errors spike, latency exceeds 3 seconds, traffic drops to zero, CPU exceeds 80%, memory exceeds 85%, or accuracy drops below 0.75.

## Key Insight
 
AWS tracks infrastructure automatically — every endpoint invocation is logged without any code. Model quality metrics require manual computation because AWS doesn't know the correct labels. We bridge this gap by computing accuracy/recall ourselves and publishing to a custom CloudWatch namespace.

In [4]:
%pip uninstall sagemaker -y
%pip install "sagemaker>=2.0,<3.0" --quiet
%pip install awswrangler scikit-learn seaborn --quiet


Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
skops 0.14.0 requires prettytable>=3.9, which is not installed.
autogluon-multimodal 1.5.0 requires fsspec[http]<=2025.3, but you have fsspec 2026.3.0 which is incompatible.
sagemaker-mlops 1.7.1 requires sagemaker-core>=2.7.1, but you have sagemaker-core 1.0.78 which is incompatible.
sagemaker-serve 1.9.0 requires sagemaker-core>=2.9.0, but you have sagemaker-core 1.0.78 which is incompatible.
sagemaker-studio-analytics-extension 0.3.0 requires sparkmagic==0.22.0, but you have spar

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [13]:
##  Setup and config 
 
import boto3, sagemaker, json, os, time
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from sagemaker import get_execution_role
from sagemaker.model_monitor import (
    DefaultModelMonitor,
    DataCaptureConfig,
    CronExpressionGenerator,
    ModelQualityMonitor,
    EndpointInput,
    MonitoringOutput,
)
from sagemaker.model_monitor.dataset_format import DatasetFormat
 
role = get_execution_role()
session = sagemaker.Session()
region = session.boto_region_name
account_id = boto3.client("sts").get_caller_identity()["Account"]
 
bucket = "pneumonia-data-set-group-4"
prefix = "monitoring"

# Clients
sm_client = boto3.client("sagemaker")
cw_client = boto3.client("cloudwatch")
s3_client = boto3.client("s3")
 
print(f"Endpoint:  {endpoint_name}")
print(f"Bucket:    {bucket}")
print(f"Region:    {region}")
print(f"Account:   {account_id}")

Endpoint:  pneumonia-onnx-2026-06-15-0417
Bucket:    pneumonia-data-set-group-4
Region:    us-east-1
Account:   134456572225


In [3]:
# Generate predictions CSV for baselines 
 
import onnxruntime as ort
from PIL import Image
import glob
 
# Load ONNX model locally
onnx_session = ort.InferenceSession("model.onnx")
input_name = onnx_session.get_inputs()[0].name
output_name = onnx_session.get_outputs()[0].name
 
# Load metadata to get ground truth labels
metadata = pd.read_csv(f"s3://{bucket}/pneumonia-project/metadata/image_metadata.csv")
print(f"Metadata loaded: {len(metadata)} rows")
print(f"Columns: {list(metadata.columns)}")
 
print(metadata.head())
print(metadata.columns.tolist())

2026-06-15 04:37:28.649048336 [W:onnxruntime:Default, device_discovery.cc:211 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:91 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


Metadata loaded: 32540 rows
Columns: ['image_id', 'raw_s3_key', 'preprocessed_s3_key', 'label', 'label_int', 'source', 'file_type', 'pixel_mean', 'pixel_std', 'img_height', 'img_width', 'event_time']
       image_id                                         raw_s3_key  \
0  IM-0001-0001  raw-images/chest-xray/test/NORMAL/IM-0001-0001...   
1  IM-0003-0001  raw-images/chest-xray/test/NORMAL/IM-0003-0001...   
2  IM-0005-0001  raw-images/chest-xray/test/NORMAL/IM-0005-0001...   
3  IM-0006-0001  raw-images/chest-xray/test/NORMAL/IM-0006-0001...   
4  IM-0007-0001  raw-images/chest-xray/test/NORMAL/IM-0007-0001...   

                           preprocessed_s3_key   label  label_int      source  \
0  preprocessed-images/NORMAL/IM-0001-0001.png  NORMAL          0  chest_xray   
1  preprocessed-images/NORMAL/IM-0003-0001.png  NORMAL          0  chest_xray   
2  preprocessed-images/NORMAL/IM-0005-0001.png  NORMAL          0  chest_xray   
3  preprocessed-images/NORMAL/IM-0006-0001.png  NORMAL 

In [4]:
# Build Predictions for Model Quality Baseline

import glob

THRESHOLD = 0.4
image_dir = "/home/sagemaker-user/preprocessed-images"

# Get all local image files
all_images = glob.glob(os.path.join(image_dir, "*.png"))
print(f"Found {len(all_images)} local images")

np.random.seed(42)
sample_images = np.random.choice(all_images, size=min(3000, len(all_images)), replace=False)

results = []
for img_path in sample_images:
    fname = os.path.basename(img_path)
    if "NORMAL" in fname:
        true_label = 0
    elif "PNEUMONIA" in fname:
        true_label = 1
    else:
        continue

    img = Image.open(img_path).convert("L").resize((128, 128))
    arr = np.array(img, dtype=np.float32)
    arr = arr.reshape(1, 128, 128, 1)

    prob = onnx_session.run([output_name], {input_name: arr})[0][0][0]
    pred_label = 1 if prob >= THRESHOLD else 0

    results.append({
        "probability": float(prob),
        "prediction": pred_label,
        "label": true_label,
    })

results_df = pd.DataFrame(results)
print(f"Generated {len(results_df)} predictions")
print(f"Prediction distribution: {results_df['prediction'].value_counts().to_dict()}")
print(f"Label distribution:      {results_df['label'].value_counts().to_dict()}")

accuracy = (results_df["prediction"] == results_df["label"]).mean()
print(f"Accuracy: {accuracy:.4f}")

Found 32540 local images


Generated 3000 predictions
Prediction distribution: {0: 1974, 1: 1026}
Label distribution:      {0: 2040, 1: 960}
Accuracy: 0.8160


In [25]:
# Upload baseline files for Model Quality Monitor 
 
model_quality_df = results_df[["probability", "prediction", "label"]].copy()
model_quality_path = "model_quality_baseline.csv"
model_quality_df.to_csv(model_quality_path, index=False, header=True)
 
s3_mq_baseline = f"{prefix}/model-quality/baseline-data/model_quality_baseline.csv"
s3_client.upload_file(model_quality_path, bucket, s3_mq_baseline)
print(f"Uploaded: s3://{bucket}/{s3_mq_baseline}")

Uploaded: s3://pneumonia-data-set-group-4/monitoring/model-quality/baseline-data/model_quality_baseline.csv


In [27]:
# Training data statistics file (for Data Quality)

feature_records = []
sample_paths = np.random.choice(all_images, size=min(2000, len(all_images)), replace=False)

for img_path in sample_paths:
    img = Image.open(img_path).convert("L").resize((128, 128))
    arr = np.array(img, dtype=np.float32)
    feature_records.append({
        "mean_pixel": float(arr.mean()),
        "std_pixel": float(arr.std()),
        "min_pixel": float(arr.min()),
        "max_pixel": float(arr.max()),
        "median_pixel": float(np.median(arr)),
    })

features_df = pd.DataFrame(feature_records)
features_path = "data_quality_baseline.csv"
features_df.to_csv(features_path, index=False, header=True)

s3_dq_baseline = f"{prefix}/data-quality/baseline-data/data_quality_baseline.csv"
s3_client.upload_file(features_path, bucket, s3_dq_baseline)
print(f"Uploaded: s3://{bucket}/{s3_dq_baseline}")
print(f"Feature stats sample:\n{features_df.describe()}")

Uploaded: s3://pneumonia-data-set-group-4/monitoring/data-quality/baseline-data/data_quality_baseline.csv
Feature stats sample:
        mean_pixel    std_pixel    min_pixel    max_pixel  median_pixel
count  2000.000000  2000.000000  2000.000000  2000.000000   2000.000000
mean    132.454915    60.082264     0.787500   244.184000    142.263500
std      16.924024     8.682905     2.014292    11.303275     21.779744
min      53.306763    34.156776     0.000000   184.000000      3.000000
25%     121.869507    53.708492     0.000000   241.000000    130.000000
50%     130.306366    61.255274     0.000000   248.000000    141.000000
75%     142.668274    66.640047     0.000000   252.000000    156.000000
max     180.836243    85.867317    39.000000   255.000000    207.000000


In [28]:
# Model Quality Monitor baseline 
 
from sagemaker.model_monitor import ModelQualityMonitor
 
model_quality_monitor = ModelQualityMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=session,
)
 
mq_baseline_uri = f"s3://{bucket}/{s3_mq_baseline}"
mq_results_uri = f"s3://{bucket}/{prefix}/model-quality/baseline-results"
 
print("Starting Model Quality Monitor baseline job...")
print("This takes ~5-10 minutes.")
 
mq_baseline_job = model_quality_monitor.suggest_baseline(
    baseline_dataset=mq_baseline_uri,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=mq_results_uri,
    problem_type="BinaryClassification",
    inference_attribute="prediction",       # column with 0/1 predictions
    probability_attribute="probability",    # column with raw probabilities
    ground_truth_attribute="label",         # column with true labels
    wait=True,
    logs=False,
)
 
print("Model Quality baseline job complete!")
print(f"Results at: {mq_results_uri}")

INFO:sagemaker:Creating processing-job with name baseline-suggestion-job-2026-06-15-03-56-42-571


Starting Model Quality Monitor baseline job...
This takes ~5-10 minutes.


.

.

.

...

.

.

.

!

Model Quality baseline job complete!
Results at: s3://pneumonia-data-set-group-4/monitoring/model-quality/baseline-results


In [29]:
## Data Quality Monitor baseline ---
 
data_quality_monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=session,
)
 
dq_baseline_uri = f"s3://{bucket}/{s3_dq_baseline}"
dq_results_uri = f"s3://{bucket}/{prefix}/data-quality/baseline-results"
 
print("Starting Data Quality Monitor baseline job...")
print("This takes ~5-10 minutes.")
 
dq_baseline_job = data_quality_monitor.suggest_baseline(
    baseline_dataset=dq_baseline_uri,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=dq_results_uri,
    wait=True,
    logs=False,
)
 
print("Data Quality baseline job complete!")
print(f"Results at: {dq_results_uri}")

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


INFO:sagemaker:Creating processing-job with name baseline-suggestion-job-2026-06-15-04-01-59-055


Starting Data Quality Monitor baseline job...
This takes ~5-10 minutes.


.

.

...

.

.

.

!

Data Quality baseline job complete!
Results at: s3://pneumonia-data-set-group-4/monitoring/data-quality/baseline-results


In [5]:
#  Download & display SageMaker reports 
 
import json
 
#  Model Quality Report 
print("=" * 60)
print("MODEL QUALITY REPORT (SageMaker Generated)")
print("=" * 60)
 
try:
    mq_statistics = model_quality_monitor.latest_baselining_job.baseline_statistics()
    print("\nModel Quality Statistics:")
    print(json.dumps(json.loads(mq_statistics.body_dict if hasattr(mq_statistics, 'body_dict')
                                else str(mq_statistics)), indent=2)[:3000])
except Exception as e:
    # Fallback: read directly from S3
    print("Reading statistics from S3...")
    stats_obj = s3_client.get_object(Bucket=bucket, Key=f"{prefix}/model-quality/baseline-results/statistics.json")
    stats = json.loads(stats_obj["Body"].read().decode())
    print(json.dumps(stats, indent=2)[:3000])
 
try:
    mq_constraints = model_quality_monitor.latest_baselining_job.suggested_constraints()
    print("\nModel Quality Constraints:")
    print(json.dumps(json.loads(mq_constraints.body_dict if hasattr(mq_constraints, 'body_dict')
                                else str(mq_constraints)), indent=2)[:3000])
except Exception as e:
    print("Reading constraints from S3...")
    cons_obj = s3_client.get_object(Bucket=bucket, Key=f"{prefix}/model-quality/baseline-results/constraints.json")
    cons = json.loads(cons_obj["Body"].read().decode())
    print(json.dumps(cons, indent=2)[:3000])
 
#  Data Quality Report 
print("\n" + "=" * 60)
print("DATA QUALITY REPORT (SageMaker Generated)")
print("=" * 60)
 
try:
    dq_statistics = data_quality_monitor.latest_baselining_job.baseline_statistics()
    print("\nData Quality Statistics:")
    print(json.dumps(json.loads(dq_statistics.body_dict if hasattr(dq_statistics, 'body_dict')
                                else str(dq_statistics)), indent=2)[:3000])
except Exception as e:
    print("Reading statistics from S3...")
    stats_obj = s3_client.get_object(Bucket=bucket, Key=f"{prefix}/data-quality/baseline-results/statistics.json")
    stats = json.loads(stats_obj["Body"].read().decode())
    print(json.dumps(stats, indent=2)[:3000])
 
try:
    dq_constraints = data_quality_monitor.latest_baselining_job.suggested_constraints()
    print("\nData Quality Constraints:")
    print(json.dumps(json.loads(dq_constraints.body_dict if hasattr(dq_constraints, 'body_dict')
                                else str(dq_constraints)), indent=2)[:3000])
except Exception as e:
    print("Reading constraints from S3...")
    cons_obj = s3_client.get_object(Bucket=bucket, Key=f"{prefix}/data-quality/baseline-results/constraints.json")
    cons = json.loads(cons_obj["Body"].read().decode())
    print(json.dumps(cons, indent=2)[:3000])

MODEL QUALITY REPORT (SageMaker Generated)
Reading statistics from S3...
{
  "version": 0.0,
  "dataset": {
    "item_count": 3000,
    "evaluation_time": "2026-06-15T04:01:08.506Z"
  },
  "binary_classification_metrics": {
    "confusion_matrix": {
      "0": {
        "0": 1731,
        "1": 309
      },
      "1": {
        "0": 243,
        "1": 717
      }
    },
    "recall": {
      "value": 0.746875,
      "standard_deviation": 0.007927105225810353
    },
    "precision": {
      "value": 0.6988304093567251,
      "standard_deviation": 0.004702496878117118
    },
    "accuracy": {
      "value": 0.816,
      "standard_deviation": 0.003707731033788163
    },
    "recall_best_constant_classifier": {
      "value": 0.0,
      "standard_deviation": 0.0
    },
    "precision_best_constant_classifier": {
      "value": 0.0,
      "standard_deviation": 0.0
    },
    "accuracy_best_constant_classifier": {
      "value": 0.68,
      "standard_deviation": 0.0039279978605329365
    },
  

{
  "version": 0.0,
  "binary_classification_constraints": {
    "recall": {
      "threshold": 0.746875,
      "comparison_operator": "LessThanThreshold"
    },
    "precision": {
      "threshold": 0.6988304093567251,
      "comparison_operator": "LessThanThreshold"
    },
    "accuracy": {
      "threshold": 0.816,
      "comparison_operator": "LessThanThreshold"
    },
    "true_positive_rate": {
      "threshold": 0.746875,
      "comparison_operator": "LessThanThreshold"
    },
    "true_negative_rate": {
      "threshold": 0.8485294117647059,
      "comparison_operator": "LessThanThreshold"
    },
    "false_positive_rate": {
      "threshold": 0.1514705882352941,
      "comparison_operator": "GreaterThanThreshold"
    },
    "false_negative_rate": {
      "threshold": 0.25312500000000004,
      "comparison_operator": "GreaterThanThreshold"
    },
    "auc": {
      "threshold": 0.8772242647058863,
      "comparison_operator": "LessThanThreshold"
    },
    "f0_5": {
      "thre

In [6]:
# Infrastructure monitors (CloudWatch alarms) 
 
# Alarm 1: High invocation errors
cw_client.put_metric_alarm(
    AlarmName=f"{endpoint_name}-high-error-rate",
    AlarmDescription="Fires when endpoint invocation errors exceed 5 in 5 minutes",
    Namespace="AWS/SageMaker",
    MetricName="Invocation5XXErrors",
    Dimensions=[{"Name": "EndpointName", "Value": endpoint_name},
                {"Name": "VariantName", "Value": "AllTraffic"}],
    Statistic="Sum",
    Period=300,
    EvaluationPeriods=1,
    Threshold=5,
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
)
print("Created alarm: High Error Rate (5XX > 5)")
 
# Alarm 2: High latency
cw_client.put_metric_alarm(
    AlarmName=f"{endpoint_name}-high-latency",
    AlarmDescription="Fires when model latency exceeds 3 seconds (p99)",
    Namespace="AWS/SageMaker",
    MetricName="ModelLatency",
    Dimensions=[{"Name": "EndpointName", "Value": endpoint_name},
                {"Name": "VariantName", "Value": "AllTraffic"}],
    ExtendedStatistic="p99",
    Period=300,
    EvaluationPeriods=2,
    Threshold=3000000,  # microseconds (3 seconds)
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
)
print("Created alarm: High Latency (p99 > 3s)")
 
# Alarm 3: Low invocations (endpoint may be down)
cw_client.put_metric_alarm(
    AlarmName=f"{endpoint_name}-low-invocations",
    AlarmDescription="Fires when no invocations for 15 minutes (endpoint health check)",
    Namespace="AWS/SageMaker",
    MetricName="Invocations",
    Dimensions=[{"Name": "EndpointName", "Value": endpoint_name},
                {"Name": "VariantName", "Value": "AllTraffic"}],
    Statistic="Sum",
    Period=900,
    EvaluationPeriods=1,
    Threshold=0,
    ComparisonOperator="LessThanOrEqualToThreshold",
    TreatMissingData="breaching",
)
print("Created alarm: Low Invocations (0 in 15 min)")
 
# Alarm 4: CPU utilization
cw_client.put_metric_alarm(
    AlarmName=f"{endpoint_name}-high-cpu",
    AlarmDescription="Fires when CPU utilization exceeds 80% for 10 minutes",
    Namespace="/aws/sagemaker/Endpoints",
    MetricName="CPUUtilization",
    Dimensions=[{"Name": "EndpointName", "Value": endpoint_name},
                {"Name": "VariantName", "Value": "AllTraffic"}],
    Statistic="Average",
    Period=300,
    EvaluationPeriods=2,
    Threshold=80,
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
)
print("Created alarm: High CPU (>80% for 10 min)")
 
# Alarm 5: Memory utilization
cw_client.put_metric_alarm(
    AlarmName=f"{endpoint_name}-high-memory",
    AlarmDescription="Fires when memory utilization exceeds 85%",
    Namespace="/aws/sagemaker/Endpoints",
    MetricName="MemoryUtilization",
    Dimensions=[{"Name": "EndpointName", "Value": endpoint_name},
                {"Name": "VariantName", "Value": "AllTraffic"}],
    Statistic="Average",
    Period=300,
    EvaluationPeriods=2,
    Threshold=85,
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
)
print("Created alarm: High Memory (>85%)")
 
# Alarm 6: Model quality  accuracy drop
cw_client.put_metric_alarm(
    AlarmName=f"{endpoint_name}-accuracy-drop",
    AlarmDescription="Custom: fires if logged accuracy drops below 0.75",
    Namespace="PneumoniaCNN/ModelQuality",
    MetricName="Accuracy",
    Statistic="Average",
    Period=3600,
    EvaluationPeriods=1,
    Threshold=0.75,
    ComparisonOperator="LessThanThreshold",
    TreatMissingData="notBreaching",
)
print("Created alarm: Accuracy Drop (<0.75)")
 
print(f"\nAll 6 infrastructure alarms created.")

Created alarm: High Error Rate (5XX > 5)
Created alarm: High Latency (p99 > 3s)
Created alarm: Low Invocations (0 in 15 min)
Created alarm: High CPU (>80% for 10 min)
Created alarm: High Memory (>85%)
Created alarm: Accuracy Drop (<0.75)

All 6 infrastructure alarms created.


In [14]:
# Publish custom model quality metrics to CloudWatch 
 
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
 
y_true = results_df["label"].values
y_pred = results_df["prediction"].values
y_prob = results_df["probability"].values
 
accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall    = recall_score(y_true, y_pred)
f1        = f1_score(y_true, y_pred)
auc       = roc_auc_score(y_true, y_prob)
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
 
metrics = [
    {"MetricName": "Accuracy",  "Value": accuracy},
    {"MetricName": "Precision", "Value": precision},
    {"MetricName": "Recall",    "Value": recall},
    {"MetricName": "F1Score",   "Value": f1},
    {"MetricName": "AUC",       "Value": auc},
    {"MetricName": "TruePositives",  "Value": float(tp)},
    {"MetricName": "FalsePositives", "Value": float(fp)},
    {"MetricName": "TrueNegatives",  "Value": float(tn)},
    {"MetricName": "FalseNegatives", "Value": float(fn)},
]
 
now = datetime.utcnow()
cw_client.put_metric_data(
    Namespace="PneumoniaCNN/ModelQuality",
    MetricData=[
        {
            "MetricName": m["MetricName"],
            "Value": m["Value"],
            "Timestamp": now,
            "Unit": "None",
            "Dimensions": [
                {"Name": "EndpointName", "Value": endpoint_name},
            ],
        }
        for m in metrics
    ],
)
 
print("Published custom metrics to CloudWatch:")
for m in metrics:
    print(f"  {m['MetricName']:20s}: {m['Value']:.4f}")

Published custom metrics to CloudWatch:
  Accuracy            : 0.8160
  Precision           : 0.6988
  Recall              : 0.7469
  F1Score             : 0.7221
  AUC                 : 0.8772
  TruePositives       : 717.0000
  FalsePositives      : 309.0000
  TrueNegatives       : 1731.0000
  FalseNegatives      : 243.0000


/tmp/ipykernel_670/3538423977.py:32: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()


In [24]:
# Endpoint Live Test: 200 Images

import random
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

test_sample = random.sample(all_images, 200)
predictor.serializer = sagemaker.serializers.IdentitySerializer(content_type="image/png")
predictor.deserializer = sagemaker.deserializers.JSONDeserializer()

ep_results = []
for i, img_path in enumerate(test_sample):
    fname = os.path.basename(img_path)
    true_label = 1 if "PNEUMONIA" in fname else 0

    with open(img_path, "rb") as f:
        response = predictor.predict(f.read())

    ep_results.append({
        "true": true_label,
        "pred": 1 if response["probability"] >= 0.4 else 0,
        "prob": response["probability"],
    })
    if (i + 1) % 50 == 0:
        print(f"Sent {i+1}/200 — {response['predicted_label']} ({response['probability']:.4f})")

ep_df = pd.DataFrame(ep_results)

ep_accuracy  = accuracy_score(ep_df["true"], ep_df["pred"])
ep_precision = precision_score(ep_df["true"], ep_df["pred"])
ep_recall    = recall_score(ep_df["true"], ep_df["pred"])
ep_f1        = f1_score(ep_df["true"], ep_df["pred"])
ep_auc       = roc_auc_score(ep_df["true"], ep_df["prob"])

print(f"\nEndpoint Accuracy:  {ep_accuracy:.4f}")
print(f"Endpoint Precision: {ep_precision:.4f}")
print(f"Endpoint Recall:    {ep_recall:.4f}")
print(f"Endpoint F1:        {ep_f1:.4f}")
print(f"Endpoint AUC:       {ep_auc:.4f}")
print(f"PNEUMONIA: {ep_df['pred'].sum()}  NORMAL: {(ep_df['pred']==0).sum()}")

Sent 50/200 — NORMAL (0.2868)


Sent 100/200 — NORMAL (0.2651)


Sent 150/200 — PNEUMONIA (0.9370)


Sent 200/200 — NORMAL (0.1239)

Endpoint Accuracy:  0.7650
Endpoint Precision: 0.5976
Endpoint Recall:    0.7778
Endpoint F1:        0.6759
Endpoint AUC:       0.8181
PNEUMONIA: 82  NORMAL: 118


In [27]:
# CloudWatch Dashboard 

dashboard_name = "Pneumonia-CNN-Monitoring"

def metric_widget(title, namespace, metric_name, stat="Average", period=300, w=12, h=6):
    return {
        "type": "metric",
        "x": 0, "y": 0, "width": w, "height": h,
        "properties": {
            "title": title,
            "metrics": [[
                namespace, metric_name,
                "EndpointName", endpoint_name,
                "VariantName", "AllTraffic",
                {"stat": stat, "period": period}
            ]],
            "view": "timeSeries",
            "region": region,
            "period": period,
        },
    }

def custom_metric_widget(title, metric_name, stat="Average", period=300, w=6, h=6):
    return {
        "type": "metric",
        "x": 0, "y": 0, "width": w, "height": h,
        "properties": {
            "title": title,
            "metrics": [[
                "PneumoniaCNN/ModelQuality", metric_name,
                "EndpointName", endpoint_name
            ]],
            "view": "singleValue",
            "region": region,
            "period": period,
            "stat": stat,
        },
    }

dashboard_body = {
    "widgets": [
        # Row 1: Infrastructure metrics
        {
            **metric_widget("Invocations", "AWS/SageMaker", "Invocations", "Sum"),
            "x": 0, "y": 0, "width": 8, "height": 6,
        },
        {
            "type": "metric",
            "x": 8, "y": 0, "width": 8, "height": 6,
            "properties": {
                "title": "Invocation Errors",
                "metrics": [
                    ["AWS/SageMaker", "Invocation4XXErrors",
                     "EndpointName", endpoint_name, "VariantName", "AllTraffic",
                     {"stat": "Sum", "period": 300, "label": "4XX Errors"}],
                    ["AWS/SageMaker", "Invocation5XXErrors",
                     "EndpointName", endpoint_name, "VariantName", "AllTraffic",
                     {"stat": "Sum", "period": 300, "label": "5XX Errors"}],
                ],
                "view": "timeSeries",
                "region": region,
            },
        },
        {
            **metric_widget("Model Latency (ms)", "AWS/SageMaker", "ModelLatency", "Average"),
            "x": 16, "y": 0, "width": 8, "height": 6,
        },

        # Row 2: Instance-level infra
        {
            **metric_widget("CPU Utilization", "/aws/sagemaker/Endpoints", "CPUUtilization"),
            "x": 0, "y": 6, "width": 8, "height": 6,
        },
        {
            **metric_widget("Memory Utilization", "/aws/sagemaker/Endpoints", "MemoryUtilization"),
            "x": 8, "y": 6, "width": 8, "height": 6,
        },
        {
            **metric_widget("Disk Utilization", "/aws/sagemaker/Endpoints", "DiskUtilization"),
            "x": 16, "y": 6, "width": 8, "height": 6,
        },

        # Row 3: Model quality metrics (custom namespace)
        {
            "type": "text",
            "x": 0, "y": 12, "width": 6, "height": 6,
            "properties": {"markdown": f"### Accuracy\n---\n# &nbsp;&nbsp;&nbsp;&nbsp;{ep_accuracy:.3f}"},
        },
        {
            "type": "text",
            "x": 6, "y": 12, "width": 6, "height": 6,
            "properties": {"markdown": f"### Precision\n---\n# &nbsp;&nbsp;&nbsp;&nbsp;{ep_precision:.3f}"},
        },
        {
            "type": "text",
            "x": 12, "y": 12, "width": 6, "height": 6,
            "properties": {"markdown": f"### Recall\n---\n# &nbsp;&nbsp;&nbsp;&nbsp;{ep_recall:.3f}"},
        },
        {
            "type": "text",
            "x": 18, "y": 12, "width": 6, "height": 6,
            "properties": {"markdown": f"### F1 Score\n---\n# &nbsp;&nbsp;&nbsp;&nbsp;{ep_f1:.3f}"},
        },

        # Row 4: AUC + Summary table
        {
            "type": "text",
            "x": 0, "y": 18, "width": 6, "height": 6,
            "properties": {"markdown": f"### AUC-ROC\n---\n# &nbsp;&nbsp;&nbsp;&nbsp;{ep_auc:.3f}"},
        },
        {
            "type": "text",
            "x": 6, "y": 18, "width": 18, "height": 6,
            "properties": {
                "markdown": f"""## Pneumonia CNN — Monitoring Summary

| Metric | Baseline (3000 local) | Endpoint (200 live) |
|--------|----------------------|---------------------|
| **Accuracy** | {accuracy:.4f} | {ep_accuracy:.4f} |
| **Precision** | {precision:.4f} | {ep_precision:.4f} |
| **Recall** | {recall:.4f} | {ep_recall:.4f} |
| **F1 Score** | {f1:.4f} | {ep_f1:.4f} |
| **AUC-ROC** | {auc:.4f} | {ep_auc:.4f} |
| **Threshold** | 0.4 | 0.4 |

**Alarms configured:** Error Rate, Latency (p99), Invocation Health, CPU, Memory, Accuracy Drop
"""
            },
        },
    ],
}

cw_client.put_dashboard(
    DashboardName=dashboard_name,
    DashboardBody=json.dumps(dashboard_body),
)

dashboard_url = f"https://{region}.console.aws.amazon.com/cloudwatch/home?region={region}#dashboards:name={dashboard_name}"
print(f"Dashboard created: {dashboard_name}")
print(f"View at: {dashboard_url}")

Dashboard created: Pneumonia-CNN-Monitoring
View at: https://us-east-1.console.aws.amazon.com/cloudwatch/home?region=us-east-1#dashboards:name=Pneumonia-CNN-Monitoring


In [45]:
# Summary of all deliverables 
 
print("=" * 65)
print("MODULE 5 DELIVERABLES — COMPLETION STATUS")
print("=" * 65)
print()
print(f"  Endpoint: {endpoint_name}")
print()
print("  1. MODEL MONITOR")
print(f"     Model Quality baseline:  s3://{bucket}/{prefix}/model-quality/baseline-results/")
print(f"     Statistics + constraints generated by SageMaker")
print()
print("  2. DATA MONITOR")
print(f"     Data Quality baseline:   s3://{bucket}/{prefix}/data-quality/baseline-results/")
print(f"     Feature statistics + constraints generated by SageMaker")
print()
print("  3. INFRASTRUCTURE MONITORS")
print(f"     6 CloudWatch alarms created:")
print(f"       - {endpoint_name}-high-error-rate")
print(f"       - {endpoint_name}-high-latency")
print(f"       - {endpoint_name}-low-invocations")
print(f"       - {endpoint_name}-high-cpu")
print(f"       - {endpoint_name}-high-memory")
print(f"       - {endpoint_name}-accuracy-drop")
print()
print("  4. CLOUDWATCH DASHBOARD")
print(f"     Dashboard: {dashboard_name}")
print(f"     URL: {dashboard_url}")
print(f"     12 widgets: invocations, errors, latency, CPU, memory,")
print(f"     disk, accuracy, precision, recall, F1, AUC, summary")
print()
print("  5. SAGEMAKER REPORTS")
print(f"     Model Quality: statistics.json + constraints.json")
print(f"     Data Quality:  statistics.json + constraints.json")
print(f"     Both at s3://{bucket}/{prefix}/")
print()
print("=" * 65)
print("All 5 deliverables complete.")
print("=" * 65)

MODULE 5 DELIVERABLES — COMPLETION STATUS

  Endpoint: pneumonia-onnx-2026-06-15-0417

  1. MODEL MONITOR
     Model Quality baseline:  s3://pneumonia-data-set-group-4/monitoring/model-quality/baseline-results/
     Statistics + constraints generated by SageMaker

  2. DATA MONITOR
     Data Quality baseline:   s3://pneumonia-data-set-group-4/monitoring/data-quality/baseline-results/
     Feature statistics + constraints generated by SageMaker

  3. INFRASTRUCTURE MONITORS
     6 CloudWatch alarms created:
       - pneumonia-onnx-2026-06-15-0417-high-error-rate
       - pneumonia-onnx-2026-06-15-0417-high-latency
       - pneumonia-onnx-2026-06-15-0417-low-invocations
       - pneumonia-onnx-2026-06-15-0417-high-cpu
       - pneumonia-onnx-2026-06-15-0417-high-memory
       - pneumonia-onnx-2026-06-15-0417-accuracy-drop

  4. CLOUDWATCH DASHBOARD
     Dashboard: Pneumonia-CNN-Monitoring
     URL: https://us-east-1.console.aws.amazon.com/cloudwatch/home?region=us-east-1#dashboards:name=

In [2]:
# Cleanup 

# Delete endpoint
sm_client.delete_endpoint(EndpointName=endpoint_name)
print(f"Deleted endpoint: {endpoint_name}")

# Delete alarms
alarm_names = [
    f"{endpoint_name}-high-error-rate",
    f"{endpoint_name}-high-latency",
    f"{endpoint_name}-low-invocations",
    f"{endpoint_name}-high-cpu",
    f"{endpoint_name}-high-memory",
    f"{endpoint_name}-accuracy-drop",
]
cw_client.delete_alarms(AlarmNames=alarm_names)
print("Deleted all alarms")

# Delete dashboard
cw_client.delete_dashboards(DashboardNames=[dashboard_name])
print(f"Deleted dashboard: {dashboard_name}")

Deleted endpoint: pneumonia-onnx-2026-06-15-0417


Deleted all alarms


Deleted dashboard: Pneumonia-CNN-Monitoring
